<a href="https://colab.research.google.com/github/FaustinoNeto/pre_data_progressao_parkinson/blob/main/pre_data_progressao_parkinson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dados de medições de proteínas e peptídeos de pacientes com Doença de Parkinson para prever a progressão da doença.

### O núcleo do conjunto de dados consiste em valores de abundância de proteínas derivados de leituras de espectrometria de massa de amostras de líquido cefalorraquidiano (CSF) coletadas de várias centenas de pacientes. Cada paciente contribuiu com várias amostras ao longo de vários anos, enquanto também realizava avaliações da gravidade da doença.

[https://www.kaggle.com/competitions/amp-parkinsons-disease-progression-prediction/data](https://)

## Conhecendo a Base De Dados:

*/content/drive/MyDrive/data-csv/train_peptides.csv*

*/content/drive/MyDrive/data-csv/train_proteins.csv*

*/content/drive/MyDrive/data-csv/supplemental_clinical_data.csv*

*/content/drive/MyDrive/data-csv/train_clinical_data.csv*

## Bibliotecas

In [ ]:
import pandas as pd
import numpy as np

# Carregamento das Bases de Dados

In [43]:
def load_data():
    from google.colab import drive
    drive.mount('/content/drive')

    train_peptides = pd.read_csv("/content/drive/MyDrive/data-csv/train_peptides.csv")
    train_proteins = pd.read_csv("/content/drive/MyDrive/data-csv/train_proteins.csv")
    train_clinical_data = pd.read_csv("/content/drive/MyDrive/data-csv/train_clinical_data.csv")
    supplemental_clinical_data = pd.read_csv("/content/drive/MyDrive/data-csv/supplemental_clinical_data.csv")
    return train_peptides, train_proteins, train_clinical_data, supplemental_clinical_data



# Análise Exploratória Inicial

In [44]:
def exploratory_analysis(df, name, head_limit=2):
    print(f"--- {name} ---")
    print(df.info())
    print(df.describe())
    print(df.head(head_limit))


# Integração dos Dados

In [34]:
def integrate_data(peptides, proteins, clinical_data, supplemental_data):
    # Garantir que a coluna 'patient_id' existe nos DataFrames
    if 'patient_id' not in peptides.columns or 'patient_id' not in proteins.columns:
        raise ValueError("A coluna 'patient_id' está ausente em um dos DataFrames fornecidos.")

    # Realizar o merge entre peptides e proteins
    merged_protein_peptide = peptides.merge(proteins, on=['visit_id', 'UniProt', 'patient_id'], how='outer')

    # Verificar a existência de 'visit_month'
    if 'visit_month_x' in merged_protein_peptide.columns and 'visit_month_y' in merged_protein_peptide.columns:
        merged_protein_peptide['visit_month'] = merged_protein_peptide['visit_month_x'].fillna(merged_protein_peptide['visit_month_y'])
        merged_protein_peptide = merged_protein_peptide.drop(['visit_month_x', 'visit_month_y'], axis=1)
    elif 'visit_month' not in merged_protein_peptide.columns:
        merged_protein_peptide['visit_month'] = np.nan

    # Integrar com clinical_data
    combined_data = merged_protein_peptide.merge(clinical_data, on='visit_id', how='outer')
    return combined_data, supplemental_data


# Tratamento de Valores Ausentes

In [35]:
def handle_missing_values(df):
    # Preenchendo valores ausentes de variáveis contínuas com mediana
    df['PeptideAbundance'] = df['PeptideAbundance'].fillna(df['PeptideAbundance'].median())
    df['NPX'] = df['NPX'].fillna(df['NPX'].median())

    # Preenchendo valores ausentes nas pontuações clínicas com mediana
    for col in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
        df[col] = df[col].fillna(df[col].median())

    # Variáveis categóricas: preenchendo com moda
    df['upd23b_clinical_state_on_medication'] = df['upd23b_clinical_state_on_medication'].fillna(df['upd23b_clinical_state_on_medication'].mode()[0])
    return df


# Normalização e Padronização

In [36]:
def normalize_data(df):
    for col in ['PeptideAbundance', 'NPX', 'updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
        df[col] = (df[col] - df[col].mean()) / df[col].std()
    return df

# Feature Engineering

In [37]:
def feature_engineering(df):
    # Agregação de abundância de peptídeos por proteína
    df['total_peptide_abundance'] = df.groupby(['UniProt'])['PeptideAbundance'].transform('sum')

    # Garantir que 'visit_month' existe e tratar valores ausentes
    if 'visit_month' not in df.columns:
        df['visit_month'] = 0
    else:
        df['visit_month'] = df['visit_month'].fillna(0)

    # Taxa de progressão da doença
    df['progression_rate'] = df['updrs_3'] / (df['visit_month'] + 1)
    return df

# Detecção e Tratamento de Outliers

In [38]:
def remove_outliers(df):
    for col in ['PeptideAbundance', 'NPX', 'updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df

# Exportação dos Dados Processados

In [47]:
def export_data(combined_data):
    combined_data.to_csv("processed_data.csv", index=False)

# Pipeline

In [49]:
def main():
    # Carregamento das Bases de Dados
    peptides, proteins, clinical_data, supplemental_data = load_data()

    # Análise Exploratória Inicial
    exploratory_analysis(peptides, "Train Peptides")
    exploratory_analysis(proteins, "Train Proteins")
    exploratory_analysis(clinical_data, "Train Clinical Data")
    exploratory_analysis(supplemental_data, "Supplemental Clinical Data")

    # Integração dos Dados
    combined_data, supplemental_data = integrate_data(peptides, proteins, clinical_data, supplemental_data)

    # Tratamento de Valores Ausentes
    combined_data = handle_missing_values(combined_data)

    # Normalização e Padronização
    combined_data = normalize_data(combined_data)

    # Feature Engineering
    combined_data = feature_engineering(combined_data)

    # Detecção e Tratamento de Outliers
    combined_data = remove_outliers(combined_data)

    # Exportação dos Dados Processados
    export_data(combined_data)


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Train Peptides ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 981834 entries, 0 to 981833
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   visit_id          981834 non-null  object 
 1   visit_month       981834 non-null  int64  
 2   patient_id        981834 non-null  int64  
 3   UniProt           981834 non-null  object 
 4   Peptide           981834 non-null  object 
 5   PeptideAbundance  981834 non-null  float64
dtypes: float64(1), int64(2), object(3)
memory usage: 44.9+ MB
None
         visit_month     patient_id  PeptideAbundance
count  981834.000000  981834.000000      9.818340e+05
mean       26.105061   32603.465361      6.428902e+05
std        22.913897   18605.934422      3.377989e+06
min         0.000000      55.000000      1.099850e+01
25%         6.00